# Step 3 — Clean, Chunk & Embed (the "Transform" stage)

We take the `RawDocument`s from Step 2 and prepare them for storage and search, in three deterministic steps:

```
RawDocument.raw_text  --clean-->  tidy text  --chunk-->  overlapping passages  --embed-->  384-dim vectors
```

- **Clean** — normalize whitespace and strip junk characters.
- **Chunk** — split into smaller overlapping passages *with a fixed rule* (no LLM).
- **Embed** — turn each passage into a vector using a **free local model** (`bge-small-en-v1.5`).

The output is a list of `Chunk` objects, each carrying its text and a 384-dimensional embedding — exactly what the Neo4j vector index from Step 1 expects.

> All code is in `signalpulse/processing.py`.

## 1. Concepts

**Why clean?** Web and PDF text arrives with stray control characters, doubled spaces, and long runs of blank lines. Cleaning makes chunking predictable and embeddings higher-quality.

**Why chunk?** Embedding models and LLMs work best on *small* passages, not whole documents. Splitting a document into passages lets semantic search return the *specific* relevant paragraph (with a citation), not a 50-page file.

- **Chunk size** — how big each passage is. We use `CHUNK_SIZE = 800` **characters** (from `.env`).
- **Chunk overlap** — how much consecutive chunks share (`CHUNK_OVERLAP = 100` chars). Overlap prevents a sentence that straddles a boundary from losing its context.
- **Recursive splitting** — `RecursiveCharacterTextSplitter` tries to break on paragraph breaks first, then lines, then sentences, then spaces — so chunks stay semantically coherent.
- **Why deterministic (not an LLM)?** A fixed rule is free, fast, and produces the *same* chunks every run — essential for an idempotent scheduled pipeline. (We *will* use the LLM in Step 4, but only for entity extraction, where it adds real value.)

**Why embed?** An **embedding** is a list of numbers capturing a passage's *meaning*. Similar meanings → nearby vectors. This powers semantic search.

- We use **`BAAI/bge-small-en-v1.5`**, run locally: **free**, no rate limits, private. It outputs **384-dimensional** vectors.
- Vectors are **L2-normalized**, so a simple dot product equals **cosine similarity** — matching the cosine vector index we built in Step 1.
- The model has a **512-token limit** per input. Our 800-char chunks stay comfortably under it — we'll *verify* this below by counting tokens.
- bge recommends a short **instruction prefix for queries** (not for passages); `embed_query()` handles that for Step 7's search.

In [1]:
import os
import sys
from pathlib import Path

# Quiet down noisy (harmless) Hugging Face warnings on Windows.
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from signalpulse import connectors as C
from signalpulse import processing as P
from signalpulse.config import settings

print(f"CHUNK_SIZE={settings.CHUNK_SIZE} chars, CHUNK_OVERLAP={settings.CHUNK_OVERLAP} chars")
print(f"Embedding model: {settings.EMBEDDING_MODEL} ({settings.EMBEDDING_DIM}-dim)")

# Fetch a few real documents to work with.
docs = C.CISAKEVConnector().fetch(limit=1)
docs += C.PDFConnector(
    "https://nvlpubs.nist.gov/nistpubs/CSWP/NIST.CSWP.29.pdf",
    name="nist_csf", agency="NIST", domain="Tech Standards & Safety", max_pages=5,
).fetch()
print("\nFetched:", [(d.connector, len(d.raw_text)) for d in docs], "(chars)")

CHUNK_SIZE=800 chars, CHUNK_OVERLAP=100 chars
Embedding model: BAAI/bge-small-en-v1.5 (384-dim)



Fetched: [('cisa_kev', 887), ('nist_csf', 8559)] (chars)


## 2. Cleaning

`clean_text()` removes control characters, converts Windows line-endings, collapses runs of spaces/tabs, caps blank-line runs, and trims each line. Below we run it on a deliberately messy string so the effect is obvious.

In [2]:
messy = "  CISA   Alert\r\n\r\n\r\n\r\n   A  critical\tvulnerability\x00 affects   systems.   \n\n\n  Patch now.  "
cleaned = P.clean_text(messy)

print("BEFORE (repr):")
print(repr(messy))
print("\nAFTER (repr):")
print(repr(cleaned))
print("\nAFTER (readable):")
print(cleaned)

BEFORE (repr):
'  CISA   Alert\r\n\r\n\r\n\r\n   A  critical\tvulnerability\x00 affects   systems.   \n\n\n  Patch now.  '

AFTER (repr):
'CISA Alert\n\nA critical vulnerability affects systems.\n\nPatch now.'

AFTER (readable):
CISA Alert

A critical vulnerability affects systems.

Patch now.


## 3. Chunking

`split_document()` cleans a document and splits it into overlapping `Chunk`s. Each chunk gets an id like `<document_id>::<index>`. Let's split the NIST PDF and inspect the result, including the **overlap** between two consecutive chunks.

In [3]:
nist_doc = docs[1]
chunks = P.split_document(nist_doc)
print(f"{nist_doc.connector}: {len(nist_doc.raw_text):,} chars -> {len(chunks)} chunks\n")

print("First chunk:")
print(" ", chunks[0].text[:300], "...\n")

# Show the overlap: the tail of chunk 0 should reappear at the head of chunk 1.
print("--- overlap check ---")
print("End of chunk 0 :", repr(chunks[0].text[-80:]))
print("Start of chunk 1:", repr(chunks[1].text[:80]))

nist_csf: 8,559 chars -> 14 chunks

First chunk:
  National Institute of Standards and Technology
This publication is available free of charge from: https://doi.org/10.6028/ NIST.CSWP.29
February 26, 2024
The NIST Cybersecurity
Framework (CSF) 2.0 ...

--- overlap check ---
End of chunk 0 : '.6028/ NIST.CSWP.29\nFebruary 26, 2024\nThe NIST Cybersecurity\nFramework (CSF) 2.0'
Start of chunk 1: 'T he NIST Cybersecurity Framework (CSF) 2.0\ni NIST C SWP 29\nFebruary 26, 2024\nAb'


### Verify chunks fit the model's 512-token limit

We chose an 800-*character* chunk size. In *tokens* (what the model actually reads), that lands well under the 512-token cap. Let's confirm by counting tokens for every chunk.

In [4]:
token_counts = [P.count_tokens(c.text) for c in chunks]
print(f"chunks: {len(chunks)}")
print(f"tokens per chunk -> min={min(token_counts)}, "
      f"max={max(token_counts)}, avg={sum(token_counts)/len(token_counts):.0f}")
print(f"model limit: 512 tokens -> all under limit: {max(token_counts) < 512}")

C:\Users\saihaj\Documents\22nd Project\.venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


chunks: 14
tokens per chunk -> min=58, max=497, avg=210
model limit: 512 tokens -> all under limit: True


## 4. Embedding

`embed_chunks()` runs the local model over every chunk and attaches a 384-dim vector. The first call loads the model (cached after the one-time ~130 MB download). Because we normalize the vectors, each one has length ≈ 1.0.

In [5]:
import numpy as np

chunks = P.embed_chunks(chunks)   # attaches .embedding to each chunk
sample = chunks[0]
vec = np.array(sample.embedding)

print(f"chunk id      : {sample.id}")
print(f"embedding dim : {len(sample.embedding)}")
print(f"vector norm   : {np.linalg.norm(vec):.4f}  (normalized -> ~1.0)")
print(f"first 8 values: {np.round(vec[:8], 4).tolist()}")

chunk id      : https://nvlpubs.nist.gov/nistpubs/CSWP/NIST.CSWP.29.pdf::0
embedding dim : 384
vector norm   : 1.0000  (normalized -> ~1.0)
first 8 values: [-0.0426, -0.022, -0.0173, -0.0141, 0.1024, 0.0345, -0.025, 0.024]


### Do the embeddings actually capture meaning?

A quick sanity check: two sentences about the *same* topic should score high cosine similarity, and an unrelated sentence should score low. Since vectors are normalized, cosine similarity is just their dot product.

In [6]:
sentences = [
    "A critical software vulnerability lets attackers run malicious code.",   # A
    "Hackers can exploit a security flaw to execute arbitrary commands.",     # B (~ A)
    "The bakery sells fresh sourdough bread every morning.",                  # C (unrelated)
]
embs = np.array(P.embed_texts(sentences))

def cos(i, j):
    return float(embs[i] @ embs[j])   # dot product = cosine (normalized)

print(f"A vs B (both about security) : {cos(0, 1):.3f}")
print(f"A vs C (security vs bakery)  : {cos(0, 2):.3f}")
print(f"B vs C (security vs bakery)  : {cos(1, 2):.3f}")
print("\n-> Related sentences score high; the unrelated one scores low. Meaning captured.")

# And how similar is our search query to the most relevant NIST chunk vs a random one?
qvec = np.array(P.embed_query("How should organizations manage cybersecurity risk?"))
chunk_vecs = np.array([c.embedding for c in chunks])
scores = chunk_vecs @ qvec
best = int(scores.argmax())
print(f"\nBest-matching chunk for the query: {chunks[best].id} (score {scores[best]:.3f})")
print("  ", chunks[best].text[:200], "...")

A vs B (both about security) : 0.803
A vs C (security vs bakery)  : 0.343
B vs C (security vs bakery)  : 0.371

-> Related sentences score high; the unrelated one scores low. Meaning captured.

Best-matching chunk for the query: https://nvlpubs.nist.gov/nistpubs/CSWP/NIST.CSWP.29.pdf::13 (score 0.795)
   online and updated regularly. Creating current and target state O rganizational Profiles helps
organizations to compare where they are versus where they want or need to be and allows them to implement ...


## 5. The full transform, in one call

`process_document(doc)` does **clean → chunk → embed** in a single step and returns embedded `Chunk`s. This is what the ingestion pipeline (Step 5) will call for every fetched document.

In [7]:
for d in docs:
    embedded = P.process_document(d)
    print(f"{d.connector:10s}: {len(d.raw_text):>6,} chars -> {len(embedded):>2} embedded chunks "
          f"(dim {len(embedded[0].embedding)})")
    print(embedded[0].preview(110))
    print()

cisa_kev  :    887 chars ->  2 embedded chunks (dim 384)
  [CVE-2026-48908::0] (emb dim=384) JoomShaper SP Page Builder Unrestricted Upload of File with Dangerous Type Vulnerability. Vendor: JoomShaper. ...



nist_csf  :  8,559 chars -> 14 embedded chunks (dim 384)
  [https://nvlpubs.nist.gov/nistpubs/CSWP/NIST.CSWP.29.pdf::0] (emb dim=384) National Institute of Standards and Technology This publication is available free of charge from: https://doi....



## Recap & what's next

We built the **Transform** stage:

- **clean** → deterministic whitespace/character normalization.
- **chunk** → `RecursiveCharacterTextSplitter`, 800-char chunks with 100-char overlap, kept under the model's 512-token limit (verified).
- **embed** → local, free `bge-small-en-v1.5` producing normalized 384-dim vectors that match our Neo4j cosine index.

We also confirmed the embeddings capture meaning (related sentences score high, unrelated low) and that a search query retrieves the most relevant chunk.

**Next — Step 4:** use the LLM to read each chunk and extract **entities and relationships** (e.g., vulnerability → affects → product), turning flat text into the knowledge graph.

> Prompt to continue:
> *"Step 4: Create notebook `04_extract_entities.ipynb`. Use the LLM (Gemini, with Groq fallback) to extract entities and relationships from chunks as structured JSON via Pydantic, so we can build the knowledge graph. Explain prompt design, structured output, and how we keep it grounded to avoid hallucination."*